In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np

from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.utils import to_categorical

with open('text', 'r', encoding='utf-8') as f:
    texts = f.read()
    texts = texts.replace('\ufeff', '')  # убираем первый невидимый символ

maxWordsCount = 1000
tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
tokenizer.fit_on_texts([texts])

dist = list(tokenizer.word_counts.items())
print(dist[:10])

data = tokenizer.texts_to_sequences([texts])
# res = to_categorical(data[0], num_classes=maxWordsCount)
# print(res.shape)
res = np.array( data[0] )

inp_words = 3
n = res.shape[0] - inp_words

X = np.array([res[i:i + inp_words] for i in range(n)])
Y = to_categorical(res[inp_words:], num_classes=maxWordsCount)

model = Sequential()
model.add(Embedding(maxWordsCount, 256, input_length = inp_words))
model.add(SimpleRNN(128, activation='tanh'))
model.add(Dense(maxWordsCount, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=50)


def buildPhrase(texts, str_len=20):
    res = texts
    data = tokenizer.texts_to_sequences([texts])[0]
    for i in range(str_len):
        # x = to_categorical(data[i: i + inp_words], num_classes=maxWordsCount)  # преобразуем в One-Hot-encoding
        # inp = x.reshape(1, inp_words, maxWordsCount)
        x = data[i: i + inp_words]
        inp = np.expand_dims(x, axis=0)

        pred = model.predict(inp)
        indx = pred.argmax(axis=1)[0]
        data.append(indx)

        res += " " + tokenizer.index_word[indx]  # дописываем строку

    return res


res = buildPhrase("позитив добавляет годы")
print(res)

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# TensorFlow-un artıq xəbərdarlıq mesajlarını gizlədir.
# DİQQƏT: TensorFlow import olunmazdan ƏVVƏL yazılmalıdır.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.array → siyahını massivə çevirmək
# - np.expand_dims → massivə yeni ölçü (batch) əlavə etmək


from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
# Keras qatları:
#
# Dense     → tam bağlı qat (çıxış qatı kimi).
# SimpleRNN → rekurrent qat, ardıcıllığı oxuyub "yaddaş" saxlayır.
# Input     → burada import olunub, amma AŞAĞIDA İSTİFADƏ OLUNMUR
#             (giriş forması Embedding-in input_length ilə verilir).
# Embedding → YENİ QAT. Hər söz nömrəsini sıx vektora çevirir
#             (aşağıda ətraflı izah olunur).


from tensorflow.keras.models import Sequential
# Sequential model: qatlar ardıcıl düzülür.


from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
# Tokenizer → mətni rəqəmlərə çevirir.
# text_to_word_sequence → mətni sözlərə bölən köməkçi funksiya.
# DİQQƏT: text_to_word_sequence aşağıda istifadə olunmur.


from tensorflow.keras.utils import to_categorical
# to_categorical → rəqəmi One-Hot vektora çevirir.
# Məsələn: 3 → [0,0,0,1,0,...]
# Burada yalnız ÇIXIŞ (Y) üçün istifadə olunur.


# =====================================================================
# BU KOD NƏDİR? — SÖZ SƏVİYYƏSİNDƏ MƏTN DAVAMI (word-level generation)
# =====================================================================
#
# Məqsəd: verilən 3 sözə baxıb NÖVBƏTİ sözü təxmin etmək,
# sonra bunu təkrarlaya-təkrarlaya cümlə yazmaq.
#
# ƏVVƏLKİ KODLA MÜQAYİSƏ (hərf səviyyəsi):
#
#                   ƏVVƏLKİ KOD            BU KOD
# Vahid:            hərf                   söz
# Cəm simvol:       34                     1000 (ən çox işlənən sözlər)
# Giriş:            6 hərf (OHE)           3 söz (nömrə)
# Giriş forması:    (6, 34)                (3,)
# Kodlama:          One-Hot                Embedding
# Çıxış:            34 sinif               1000 sinif
#
# ƏSAS FƏRQ: söz səviyyəsində OHE çox böyük olardı (1000 ölçülü
# vektor × 3 söz). Ona görə girişdə OHE əvəzinə Embedding var.


with open('text', 'r', encoding='utf-8') as f:
    texts = f.read()
    texts = texts.replace('\ufeff', '')  # убираем первый невидимый символ
# 'text' adlı faylı oxuyur (kodla eyni qovluqda olmalıdır).
# '\ufeff' → görünməyən BOM simvolu, silirik.
#
# Əvvəlki koddan fərq: burada re.sub ilə təmizləmə YOXDUR.
# Durğu işarələrini Tokenizer-in "filters" parametri təmizləyir.


maxWordsCount = 1000
# Lüğətin maksimum ölçüsü: ən çox işlənən 1000 söz nəzərə alınır.
# Qalan (nadir) sözlər atılır.
# Bu rəqəm həm Embedding-in giriş ölçüsü, həm də Dense-in
# çıxış ölçüsüdür.

tokenizer = Tokenizer(num_words=maxWordsCount, filters='!–"—#$%&amp;()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r«»',
                      lower=True, split=' ', char_level=False)
# Söz səviyyəsində Tokenizer.
#
# num_words=1000 → ən çox işlənən 1000 sözü saxla.
#
# filters='...' → mətndən silinəcək simvollar (durğu işarələri,
#   xüsusi simvollar, sətir keçidi, tire, dırnaqlar və s.).
#   DİQQƏT: '&amp;' HTML-dən qalıb. Bu o deməkdir ki, '&', 'a', 'm',
#   'p', ';' simvolları da filtrə düşür. Rus mətnində problem yoxdur,
#   amma latın hərfli mətndə 'a', 'm', 'p' silinərdi.
#   Düzgün variant: '&' yazmaq.
#
# lower=True → bütün sözləri kiçik hərfə çevirir
#   ("Позитив" və "позитив" eyni söz sayılır).
#
# split=' ' → sözləri boşluğa görə ayırır.
#
# char_level=False → HƏRF yox, SÖZ səviyyəsində işlə.
#   (Əvvəlki kodda True idi.)

tokenizer.fit_on_texts([texts])
# Mətni oxuyub hər sözə nömrə verir.
# Ən çox işlənən söz → 1, sonrakı → 2 və s.
# 0 nömrəsi boş saxlanılır.

dist = list(tokenizer.word_counts.items())
print(dist[:10])
# word_counts → hər sözün mətndə neçə dəfə işləndiyi lüğət.
# .items() → (söz, say) cütləri, list() → siyahıya çevirir.
# İlk 10 cütü çap edir. Məsələn: [('позитив', 5), ('добавляет', 2), ...]
# Bu sıralama tezliyə görə DEYİL, mətndə ilk rast gəlmə ardıcıllığına görədir.


data = tokenizer.texts_to_sequences([texts])
# Mətni söz nömrələrinin siyahısına çevirir.
# Nəticə: siyahı içində siyahı, məsələn [[45, 12, 7, 3, ...]]
#
# DİQQƏT: nömrəsi 1000 və daha böyük olan (nadir) sözlər
# nəticəyə düşmür, sadəcə atılır.

# res = to_categorical(data[0], num_classes=maxWordsCount)
# print(res.shape)
# Bu iki sətir şərhə alınıb. Bütün mətni OHE etmək idi
# (əvvəlki kod belə edirdi), amma 1000 ölçülü vektor çox yaddaş tutar.

res = np.array(data[0])
# İç siyahını götürüb massivə çevirir.
# Forma: (sözlərin_sayı,) — sadə nömrələr massivi.
# OHE yoxdur, hər söz bir tam ədəddir.


inp_words = 3
# Model hər dəfə 3 sözə baxıb 4-cü sözü təxmin edir.
# (Əvvəlki kodda inp_chars=6 idi.)

n = res.shape[0] - inp_words
# Nümunə sayı: sonuncu 3 sözdən sonra təxmin ediləcək söz yoxdur.

X = np.array([res[i:i + inp_words] for i in range(n)])
# "Sürüşən pəncərə" üsulu: hər dəfə 3 ardıcıl söz.
#
# Məsələn, nömrələr [5, 9, 2, 7, 4]:
#   i=0 → [5, 9, 2]
#   i=1 → [9, 2, 7]
#
# Forma: (n, 3) — yalnız nömrələr (OHE olsaydı (n, 3, 1000) olardı).

Y = to_categorical(res[inp_words:], num_classes=maxWordsCount)
# Düzgün cavablar: hər pəncərədən sonra gələn söz.
#   X[0] = [5, 9, 2] → Y[0] = söz 7
#
# to_categorical → cavabı OHE vektora çevirir: forma (n, 1000).
# Softmax + categorical_crossentropy üçün lazımdır.
#
# DİQQƏT: n × 1000 massivi böyük yaddaş tutur. Məsələn, 100 min
# söz üçün ≈ 100000 × 1000 × 4 bayt ≈ 400 MB. Mətn böyükdürsə
# yaddaş çatışmaya bilər.


model = Sequential()

model.add(Embedding(maxWordsCount, 256, input_length=inp_words))
# EMBEDDING QATI — bu kodun ən vacib yeniliyi.
#
# Nə edir? Hər söz nömrəsini 256 ədədlik sıx vektora çevirir.
#
# Niyə OHE yox? OHE-də hər söz 1000 ölçülü vektordur, yalnız bir
# yerdə 1, qalanı 0 (israfçılıq). Üstəlik OHE-də bütün sözlər
# bir-birindən eyni dərəcədə "uzaq"dır.
# Embedding-də isə sözün vektoru ÖYRƏNİLİR: mənaca yaxın sözlər
# zamanla yaxın vektorlar alır.
#
# Parametrlər:
#   1000 → lüğət ölçüsü (neçə fərqli nömrə ola bilər)
#   256  → hər sözün vektor ölçüsü
#   input_length=3 → giriş ardıcıllığının uzunluğu
#
# Forma: (None, 3) → (None, 3, 256)
# Parametr sayı: 1000 × 256 = 256000
#
# DİQQƏT: input_length arqumenti yeni Keras 3 versiyasında
# silinib, xəta verə bilər. Onda bu arqumenti çıxarıb əvvəldə
# model.add(Input((inp_words,))) yazmaq lazımdır.

model.add(SimpleRNN(128, activation='tanh'))
# Rekurrent qat, 128 neyron.
# 3 sözü ardıcıl oxuyur, hər addımda əvvəlki sözləri xatırlayır.
# Yalnız son vəziyyəti çıxışa verir.
#
# Forma: (3, 256) → (128,)
# Parametr sayı:
#   giriş çəkiləri:   256 × 128 = 32768
#   yaddaş çəkiləri:  128 × 128 = 16384
#   bias:                         128
#   cəmi:                       49280

model.add(Dense(maxWordsCount, activation='softmax'))
# Çıxış qatı: 1000 neyron, hər söz üçün bir ehtimal.
# softmax → ehtimalların cəmi 1 olur.
#
# Forma: (128,) → (1000,)
# Parametr: 128 × 1000 + 1000 = 129000

model.summary()
# Modelin quruluşunu çap edir.
# Ümumi parametr: 256000 + 49280 + 129000 = 434280.
# Əvvəlki hərf modelindən (25250) təxminən 17 dəfə böyükdür.

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
# loss → çoxsinifli təsnifat itkisi (burada 1000 sinif).
# metrics → düzgün təxmin faizi.
# optimizer='adam' → çəkiləri avtomatik yeniləyir.

history = model.fit(X, Y, batch_size=32, epochs=50)
# Öyrətmə: hər addımda 32 nümunə, bütün data 50 dəfə keçilir.
# history → hər epoch-un loss və accuracy tarixçəsi.
#
# Qeyd: söz səviyyəsində dəqiqlik adətən aşağı olur, çünki 1000
# variant arasından seçmək hərf seçməkdən (34 variant) çətindir.
# Kiçik mətndə model əzbərləməyə (overfitting) meyilli olur.


def buildPhrase(texts, str_len=20):
    # Başlanğıc ifadəni götürüb ona str_len (20) söz əlavə edir.

    res = texts
    # Nəticə sətri: başlanğıc ifadə ilə başlayır.

    data = tokenizer.texts_to_sequences([texts])[0]
    # Başlanğıc ifadəni nömrələr siyahısına çevirir.
    # "позитив добавляет годы" → məsələn [45, 87, 120].
    # [0] → iç siyahını götürür.
    # DİQQƏT: ifadədə dəqiq 3 söz olmalıdır (inp_words=3).
    # Lüğətdə olmayan söz atılsa, siyahı 3-dən qısa qalar və xəta çıxar.

    for i in range(str_len):
        # 20 dəfə təkrarlayır, hər dəfə bir yeni söz.

        # x = to_categorical(data[i: i + inp_words], num_classes=maxWordsCount)  # преобразуем в One-Hot-encoding
        # inp = x.reshape(1, inp_words, maxWordsCount)
        # Köhnə OHE variantı, şərhə alınıb. Embedding ilə lazım deyil.

        x = data[i: i + inp_words]
        # Siyahının son 3 nömrəsi (pəncərə).
        # i artdıqca pəncərə sürüşür, çünki data-ya yeni nömrələr
        # əlavə olunur.

        inp = np.expand_dims(x, axis=0)
        # Forma (3,) → (1, 3). Əvvəldə batch ölçüsü əlavə edir.
        # Model həmişə nümunələr toplusu gözləyir; bizdə 1 nümunə var.

        pred = model.predict(inp)
        # Model 1000 söz üçün ehtimal verir. Forma: (1, 1000).

        indx = pred.argmax(axis=1)[0]
        # Ən yüksək ehtimallı sözün nömrəsi.
        # DİQQƏT: argmax həmişə eyni cavabı verir (təsadüfilik yox),
        # buna görə model bəzən eyni sözləri təkrarlayıb dövrəyə düşə bilər.

        data.append(indx)
        # Təxmin edilən nömrəni siyahının sonuna əlavə edir ki,
        # növbəti addımda pəncərəyə düşsün.

        res += " " + tokenizer.index_word[indx]  # дописываем строку
        # Nömrəni sözə çevirib sətrin sonuna əlavə edir.
        # index_word → nömrədən sözü tapan lüğət.

    return res
    # Yekun: başlanğıc 3 söz + 20 yeni söz.


res = buildPhrase("позитив добавляет годы")
# Başlanğıc: 3 söz ("позитив добавляет годы").
# Model növbəti sözü təxmin edir, sonra
# "добавляет годы X" → Y və s., 20 söz yazır.

print(res)
# Nəticəni çap edir. Söz birləşmələri çox vaxt dil baxımından
# məntiqli görünür, amma mənası zəif ola bilər, çünki yalnız
# 3 sözlük kontekstə baxır.